In [1]:
# --- Cell 1: imports and environment check ---

import os
import sys
import time
import csv
import gc
import platform
from datetime import datetime

import numpy as np
import pandas as pd

try:
    import jax
    import jax.numpy as jnp
    print("JAX devices:", jax.devices())
except Exception as exc:
    print("JAX import/device check failed:", repr(exc))

import WDM

from few.waveform import GenerateEMRIWaveform
from few.utils.constants import YRSID_SI

# Adjust these imports if your local package path differs.
from WDM_LISAresponse.code.lisa_response.waveforms import generate_windowed_padded_waveform
from WDM_LISAresponse.code.lisa_response.response_sub import build_sky_basis
from WDM_LISAresponse.code.lisa_response.response import (
    one_arm_response_wdm_dT,
    _build_wdm_dT_context,
    _one_arm_wdm_dT_components_from_context,
    _one_arm_response_wdm_dT_from_context,
)
from WDM.code.time_delay_filters.time_shift_fast import wdm_time_shift_variable

# Your orbit import/setup may differ.
# Keep whatever worked in the previous notebook.
# Example:
# from lisatools.detector import ESAOrbits

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Current working directory:", os.getcwd())

JAX devices: [CpuDevice(id=0)]
Python executable: /Users/tomekkinowski/opt/anaconda3/envs/lisatools/bin/python
Python version: 3.13.11 | packaged by conda-forge | (main, Dec  6 2025, 11:37:04) [Clang 19.1.7 ]
Platform: macOS-26.2-x86_64-i386-64bit-Mach-O
Current working directory: /Users/tomekkinowski/PhD_1/LISA_Response/WDM_LISAresponse/examples


In [2]:
# --- Cell 2: timing helpers ---

def time_call(fn, warm=0, repeat=1, label=None, collect=True):
    """
    Time a callable.

    For expensive response code, use warm=0, repeat=1.
    For isolated JAX kernels, warm=1 can be useful if you want post-compile timing.
    """
    if collect:
        gc.collect()

    for _ in range(max(0, int(warm))):
        _ = fn()

    if collect:
        gc.collect()

    t0 = time.perf_counter()
    out = None
    for _ in range(max(1, int(repeat))):
        out = fn()
    t1 = time.perf_counter()

    avg = (t1 - t0) / max(1, int(repeat))

    if label is not None:
        print(f"{label}: {avg:.6f} s")

    return out, avg


def array_mb(x):
    x = np.asarray(x)
    return x.nbytes / 1024**2


def print_table(rows, columns):
    widths = [len(col) for col in columns]
    for row in rows:
        for i, col in enumerate(columns):
            widths[i] = max(widths[i], len(str(row.get(col, ""))))
    header = "  ".join(col.ljust(widths[i]) for i, col in enumerate(columns))
    print(header)
    print("-" * len(header))
    for row in rows:
        print("  ".join(str(row.get(col, "")).ljust(widths[i]) for i, col in enumerate(columns)))


def summarize_array(name, x):
    x = np.asarray(x)
    return {
        "name": name,
        "shape": str(x.shape),
        "dtype": str(x.dtype),
        "MB": f"{array_mb(x):.2f}",
        "finite": str(np.all(np.isfinite(x))),
    }

In [3]:
# --- Cell 3: profiling configuration ---

# Main profiling choice.
DURATION_YR = 1.0      # start modest; raise after the notebook is stable
dt = 10
Nf = 100
q_wdm = 4
d_wdm = 4
L_trunc_value = 25

# Link to profile.
s_link = 1
r_link = 2

# Sky/source.
theta = np.pi / 2
phi = 0.0

# Window.
alpha = 0.01
tukey_factor = 5

# Fast shift mode to profile.
variable_shift_kwargs = {
    "assembly_backend": "lagfirst_chunked",
    "assembly_precision": "complex64",
    "row_chunk_size": 128,
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
}

# Set this False for normal user-facing timing.
# Set True only if you intentionally want time-domain reconstruction too.
return_td = False

print("Profiling configuration:")
print("DURATION_YR:", DURATION_YR)
print("dt:", dt)
print("Nf:", Nf)
print("q_wdm:", q_wdm)
print("d_wdm:", d_wdm)
print("L_trunc_value:", L_trunc_value)
print("link:", (s_link, r_link))
print("variable_shift_kwargs:", variable_shift_kwargs)

Profiling configuration:
DURATION_YR: 1.0
dt: 10
Nf: 100
q_wdm: 4
d_wdm: 4
L_trunc_value: 25
link: (1, 2)
variable_shift_kwargs: {'assembly_backend': 'lagfirst_chunked', 'assembly_precision': 'complex64', 'row_chunk_size': 128, 'tl_tp_mode': 'interp', 'tl_tp_interp_points': 32, 'tl_tp_interp_kind': 'linear'}


In [4]:
# --- Cell 4: orbit setup ---

from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

print("Loading orbit file...")

orbit_data = read_orbit_file("esa-trailing-orbits.h5")

orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)

orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

# Match the benchmark notebook convention.
t0 = orbit_data.t0 + 0.1 * YRSID_SI

print("Loaded orbits.")
print("orbit_data.t0:", orbit_data.t0)
print("orbit_data.dt:", orbit_data.dt)
print("orbit_data.size:", orbit_data.size)
print("orbit_times shape:", orbit_times.shape)
print("positions shape:", orbit_data.positions.shape)
print("velocities shape:", orbit_data.velocities.shape)
print("orbits_trailing:", type(orbits_trailing))
print("t0:", t0)

Loading orbit file...
Loaded orbits.
orbit_data.t0: 2073211130.8175
orbit_data.dt: 161789.95624780655
orbit_data.size: 1721
orbit_times shape: (1721,)
positions shape: (1721, 3, 3)
velocities shape: (1721, 3, 3)
orbits_trailing: <class 'lisaorbits.orbits.InterpolatedOrbits'>
t0: 2076366945.7938547


In [5]:
# --- Cell 5: generate waveform once ---

td_gen = GenerateEMRIWaveform(
    "FastKerrEccentricEquatorialFlux",
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

# EMRI parameters copied from benchmark notebook.
dist = 1.0
m1 = 1e6
m2 = 10.0
a = 0.9
p0 = 8.5
e0 = 0.3
x0 = 1.0
qK = np.pi / 3
phiK = np.pi / 3
eps = 1e-2
Phi_phi0 = np.pi / 3
Phi_theta0 = 0.0
Phi_r0 = np.pi / 3

model_kwargs = {
    "td_gen": td_gen,
    "m1": m1,
    "m2": m2,
    "a": a,
    "p0": p0,
    "e0": e0,
    "x0": x0,
    "dist": dist,
    "theta": theta,
    "phi": phi,
    "qK": qK,
    "phiK": phiK,
    "Phi_phi0": Phi_phi0,
    "Phi_theta0": Phi_theta0,
    "Phi_r0": Phi_r0,
    "eps": eps,
}

window_kwargs = {
    "alpha": alpha,
    "tukey_factor": tukey_factor,
}

T_obs = DURATION_YR * YRSID_SI
N = int(np.round(T_obs / dt))

print("Generating waveform...")
print("DURATION_YR:", DURATION_YR)
print("dt:", dt)
print("N:", N)
print("Nf:", Nf)

(times_pad, hplus_td, hcross_td, pad_info), waveform_time_s = time_call(
    lambda: generate_windowed_padded_waveform(
        waveform_model="emri",
        t0=t0,
        dt=dt,
        N=N,
        model_kwargs=model_kwargs,
        window_function="tukey",
        window_kwargs=window_kwargs,
        wdm_Nf=Nf,
        return_padding_info=True,
    ),
    warm=0,
    repeat=1,
    label="waveform_generation",
)

print("\nWaveform generated.")
print("times_pad shape:", times_pad.shape)
print("hplus_td shape:", hplus_td.shape)
print("hcross_td shape:", hcross_td.shape)
print("times_pad duration days:", (times_pad[-1] - times_pad[0]) / 86400.0)
print("pad_info:", pad_info)

print("\nArray summaries:")
print_table(
    [
        summarize_array("times_pad", times_pad),
        summarize_array("hplus_td", hplus_td),
        summarize_array("hcross_td", hcross_td),
    ],
    ["name", "shape", "dtype", "MB", "finite"],
)

Generating waveform...
DURATION_YR: 1.0
dt: 10
N: 3155815
Nf: 100
waveform_generation: 17.013407 s

Waveform generated.
times_pad shape: (3313800,)
hplus_td shape: (3313800,)
hcross_td shape: (3313800,)
times_pad duration days: 383.54155092592595
pad_info: {'original_length': 3155815, 'window_pad_left': 78896, 'window_pad_right': 78896, 'window_function': 'tukey', 'wdm_Nf': 100, 'wdm_tail_pad_right': 193, 'wdm_target_length': 3313800, 'wdm_Nt': 33138}

Array summaries:
name       shape       dtype    MB     finite
---------------------------------------------
times_pad  (3313800,)  float64  25.28  True  
hplus_td   (3313800,)  float64  25.28  True  
hcross_td  (3313800,)  float64  25.28  True  


In [6]:
# --- Cell 6: public one-shot one-arm timing ---

def run_public_one_arm():
    return one_arm_response_wdm_dT(
        times_pad,
        s_link,
        r_link,
        orbits_trailing,
        theta,
        phi,
        hplus_td,
        hcross_td,
        Nf=Nf,
        return_td=return_td,
        d=d_wdm,
        q=q_wdm,
        L_trunc=L_trunc_value,
        variable_shift_kwargs=variable_shift_kwargs,
    )

w_public, public_time_s = time_call(
    run_public_one_arm,
    warm=0,
    repeat=1,
    label="public_one_arm_response_wdm_dT",
)

print("\nPublic one-arm result:")
print("output shape:", np.asarray(w_public).shape)
print("output dtype:", np.asarray(w_public).dtype)
print("output MB:", f"{array_mb(w_public):.2f}")
print("finite:", np.all(np.isfinite(w_public)))

public_one_arm_response_wdm_dT: 15.329776 s

Public one-arm result:
output shape: (33138, 100)
output dtype: complex64
output MB: 25.28
finite: True


In [7]:
# --- Cell 7: stage A: build WDM context / DWT ---

def run_build_context():
    return _build_wdm_dT_context(
        times_pad,
        hplus_td,
        hcross_td,
        Nf=Nf,
        d=d_wdm,
        q=q_wdm,
        calc_m0=True,
    )

context, context_time_s = time_call(
    run_build_context,
    warm=0,
    repeat=1,
    label="stage_A_build_context_DWT",
)

wdm = context["wdm"]

print("\nWDM object:")
print("  Nt:", wdm.Nt)
print("  Nf:", wdm.Nf)
print("  N:", wdm.N)
print("  dt:", wdm.dt)
print("  dT:", wdm.dT)
print("  dF:", wdm.dF)
print("  q:", getattr(wdm, "q", None))
print("  calc_m0:", getattr(wdm, "calc_m0", None))

print("\nContext array summaries:")
context_arrays = [
    summarize_array("w_t_plus", context["w_t_plus"]),
    summarize_array("w_t_cross", context["w_t_cross"]),
    summarize_array("freq_vals", context["freq_vals"]),
    summarize_array("int_times_vals", context["int_times_vals"]),
]
print_table(context_arrays, ["name", "shape", "dtype", "MB", "finite"])

stage_A_build_context_DWT: 8.140942 s

WDM object:
  Nt: 33138
  Nf: 100
  N: 3313800
  dt: 10.0
  dT: 1000.0
  dF: 0.0005
  q: 4
  calc_m0: True

Context array summaries:
name            shape         dtype    MB     finite
----------------------------------------------------
w_t_plus        (33138, 100)  float64  25.28  True  
w_t_cross       (33138, 100)  float64  25.28  True  
freq_vals       (100,)        float64  0.00   True  
int_times_vals  (33138,)      float64  0.25   True  


In [8]:
# --- Cell 8: stage B: one-arm geometry + w_xi construction ---

sky_basis = build_sky_basis(theta, phi)

def run_components():
    return _one_arm_wdm_dT_components_from_context(
        context,
        s_link,
        r_link,
        orbits_trailing,
        theta,
        phi,
        include_time_domain_geometry=False,
        sky_basis=sky_basis,
    )

(xi_s, tau_s, w_xi, time_shift), components_time_s = time_call(
    run_components,
    warm=0,
    repeat=1,
    label="stage_B_components_geometry_wxi",
)

print("\nComponent result summaries:")
component_arrays = [
    summarize_array("w_xi", w_xi),
    summarize_array("time_shift", time_shift),
]
print_table(component_arrays, ["name", "shape", "dtype", "MB", "finite"])

print("\nExtra outputs:")
print("xi_s is None:", xi_s is None)
print("tau_s is None:", tau_s is None)

print("\ntime_shift stats:")
print("  min:", float(np.min(time_shift)))
print("  max:", float(np.max(time_shift)))
print("  mean:", float(np.mean(time_shift)))
print("  std:", float(np.std(time_shift)))

stage_B_components_geometry_wxi: 0.120903 s

Component result summaries:
name        shape         dtype    MB     finite
------------------------------------------------
w_xi        (33138, 100)  float64  25.28  True  
time_shift  (33138,)      float64  0.25   True  

Extra outputs:
xi_s is None: True
tau_s is None: True

time_shift stats:
  min: -493.52733647133164
  max: 503.87794631795106
  mean: 29.571856784711496
  std: 359.86406205301813


In [9]:
# --- Cell 9: stage C: final coefficient-space variable time shift ---

def run_final_shift():
    return wdm_time_shift_variable(
        wdm,
        w_xi,
        -time_shift,
        Nf=wdm.Nf,
        delta_mode="target",
        L_trunc=L_trunc_value,
        **variable_shift_kwargs,
    )

# Cold timing: includes any first-call JAX compilation/cache/setup overhead.
w_shifted_cold, final_shift_cold_time_s = time_call(
    run_final_shift,
    warm=0,
    repeat=1,
    label="stage_C_final_time_shift_cold",
)

# Warm timing: one warm run first, then measure one apply.
w_shifted, final_shift_warm_time_s = time_call(
    run_final_shift,
    warm=1,
    repeat=1,
    label="stage_C_final_time_shift_warm",
)

print("\nShifted result:")
print("shifted shape:", np.asarray(w_shifted).shape)
print("shifted dtype:", np.asarray(w_shifted).dtype)
print("shifted MB:", f"{array_mb(w_shifted):.2f}")
print("finite:", np.all(np.isfinite(w_shifted)))

rel_cold_warm = (
    np.linalg.norm(np.asarray(w_shifted_cold) - np.asarray(w_shifted))
    / np.linalg.norm(np.asarray(w_shifted))
)
print("rel_l2 cold vs warm:", rel_cold_warm)

stage_C_final_time_shift_cold: 0.956655 s
stage_C_final_time_shift_warm: 0.884117 s

Shifted result:
shifted shape: (33138, 100)
shifted dtype: complex64
shifted MB: 25.28
finite: True
rel_l2 cold vs warm: 0.0


In [10]:
# --- Cell 10: context-reused one-arm timing ---

def run_context_reused_one_arm():
    return _one_arm_response_wdm_dT_from_context(
        context,
        s_link,
        r_link,
        orbits_trailing,
        theta,
        phi,
        return_td=False,
        L_trunc=L_trunc_value,
        variable_shift_kwargs=variable_shift_kwargs,
        sky_basis=sky_basis,
    )

w_context_reused, context_reused_time_s = time_call(
    run_context_reused_one_arm,
    warm=0,
    repeat=1,
    label="context_reused_one_arm",
)

print("\nContext-reused result:")
print("output shape:", np.asarray(w_context_reused).shape)
print("output dtype:", np.asarray(w_context_reused).dtype)
print("output MB:", f"{array_mb(w_context_reused):.2f}")
print("finite:", np.all(np.isfinite(w_context_reused)))

if "w_public" in globals():
    rel_public = (
        np.linalg.norm(np.asarray(w_context_reused) - np.asarray(w_public))
        / np.linalg.norm(np.asarray(w_public))
    )
    print("rel_l2 context_reused vs public:", rel_public)

if "w_shifted" in globals():
    rel_shift = (
        np.linalg.norm(np.asarray(w_context_reused) - np.asarray(w_shifted))
        / np.linalg.norm(np.asarray(w_shifted))
    )
    print("rel_l2 context_reused vs manual components+shift:", rel_shift)

context_reused_one_arm: 1.066726 s

Context-reused result:
output shape: (33138, 100)
output dtype: complex64
output MB: 25.28
finite: True
rel_l2 context_reused vs public: 0.0
rel_l2 context_reused vs manual components+shift: 0.0


In [11]:
# --- Cell 11: timing summary ---

rows_profile = []

rows_profile.append({
    "stage": "waveform_generation",
    "time_s": f"{waveform_time_s:.4f}" if "waveform_time_s" in globals() else "nan",
    "notes": "FEW waveform + window/padding",
})

rows_profile.append({
    "stage": "public_one_arm_total",
    "time_s": f"{public_time_s:.4f}" if "public_time_s" in globals() else "nan",
    "notes": "one_arm_response_wdm_dT one-shot",
})

rows_profile.append({
    "stage": "A_context_DWT",
    "time_s": f"{context_time_s:.4f}" if "context_time_s" in globals() else "nan",
    "notes": "WDM object + dwt(hplus/hcross)",
})

rows_profile.append({
    "stage": "B_components_geometry_wxi",
    "time_s": f"{components_time_s:.4f}" if "components_time_s" in globals() else "nan",
    "notes": "orbit/LTT/antenna/sinc/w_xi",
})

rows_profile.append({
    "stage": "C_final_time_shift_cold",
    "time_s": f"{final_shift_cold_time_s:.4f}" if "final_shift_cold_time_s" in globals() else "nan",
    "notes": "wdm_time_shift_variable first call",
})

rows_profile.append({
    "stage": "C_final_time_shift_warm",
    "time_s": f"{final_shift_warm_time_s:.4f}" if "final_shift_warm_time_s" in globals() else "nan",
    "notes": "wdm_time_shift_variable after warm run",
})

rows_profile.append({
    "stage": "context_reused_one_arm",
    "time_s": f"{context_reused_time_s:.4f}" if "context_reused_time_s" in globals() else "nan",
    "notes": "components + shift with context reused",
})

if all(k in globals() for k in ["context_time_s", "components_time_s", "final_shift_warm_time_s"]):
    staged_sum_warm = context_time_s + components_time_s + final_shift_warm_time_s
    rows_profile.append({
        "stage": "A+B+C_warm_sum",
        "time_s": f"{staged_sum_warm:.4f}",
        "notes": "approx staged sum using warm shift",
    })

if all(k in globals() for k in ["context_time_s", "components_time_s", "final_shift_cold_time_s"]):
    staged_sum_cold = context_time_s + components_time_s + final_shift_cold_time_s
    rows_profile.append({
        "stage": "A+B+C_cold_sum",
        "time_s": f"{staged_sum_cold:.4f}",
        "notes": "approx staged sum using cold shift",
    })

print("Timing summary:")
print_table(rows_profile, ["stage", "time_s", "notes"])

if "public_time_s" in globals() and public_time_s > 0:
    print("\nPercent of public one-arm total:")
    for row in rows_profile:
        try:
            t = float(row["time_s"])
            print(f"  {row['stage']}: {100.0 * t / public_time_s:.1f}%")
        except Exception:
            pass

Timing summary:
stage                      time_s   notes                                 
--------------------------------------------------------------------------
waveform_generation        17.0134  FEW waveform + window/padding         
public_one_arm_total       15.3298  one_arm_response_wdm_dT one-shot      
A_context_DWT              8.1409   WDM object + dwt(hplus/hcross)        
B_components_geometry_wxi  0.1209   orbit/LTT/antenna/sinc/w_xi           
C_final_time_shift_cold    0.9567   wdm_time_shift_variable first call    
C_final_time_shift_warm    0.8841   wdm_time_shift_variable after warm run
context_reused_one_arm     1.0667   components + shift with context reused
A+B+C_warm_sum             9.1460   approx staged sum using warm shift    
A+B+C_cold_sum             9.2185   approx staged sum using cold shift    

Percent of public one-arm total:
  waveform_generation: 111.0%
  public_one_arm_total: 100.0%
  A_context_DWT: 53.1%
  B_components_geometry_wxi: 0.8%
  C_fin